# 📐 Mathematics Used in Large Language Models

This notebook walks through every core mathematical concept that powers modern LLMs —
from the softmax function to positional encodings — with worked examples and visualisations.

**Topics covered:**
1. Scalars, vectors, matrices
2. Dot products and cosine similarity
3. Softmax and log-softmax
4. Cross-entropy loss
5. Scaled dot-product attention
6. Layer normalisation
7. Positional encoding (sinusoidal)
8. Gradient descent intuition
9. Temperature sampling
10. Perplexity

In [1]:
# ── Imports ──────────────────────────────────────────────────────────────────
# numpy  : numerical computing backbone
# matplotlib : plotting and visualisation
import numpy as np
import matplotlib.pyplot as plt
import warnings

# Suppress minor warnings for cleaner notebook output
warnings.filterwarnings('ignore')

# Set a consistent random seed so all examples are reproducible
np.random.seed(42)

print('Libraries loaded successfully.')

Libraries loaded successfully.


## 1. Scalars, Vectors, Matrices — The Building Blocks

LLMs represent text as **token embeddings** — high-dimensional vectors.
All computations (attention, FFN, projections) are matrix operations.

In [2]:
# ── Scalars, Vectors, Matrices ────────────────────────────────────────────────

# Scalar: a single number — e.g. a learning rate
learning_rate = 3e-4
print(f'Scalar (learning rate): {learning_rate}')

# Vector: a 1-D array — e.g. a token embedding of dimension d=4
token_embedding = np.array([0.2, -0.5, 0.8, 0.1])
print(f'Token embedding (d=4): {token_embedding}')

# Matrix: a 2-D array — e.g. an embedding table (vocab x d_model)
# Here vocab=5 tokens, d_model=4 dimensions
embedding_table = np.random.randn(5, 4)  # shape (V, d)
print(f'Embedding table shape: {embedding_table.shape}')  # (5, 4)

# Matrix × vector: look up row 2 of the embedding table
one_hot = np.array([0, 0, 1, 0, 0])       # one-hot for token id=2
looked_up = embedding_table.T @ one_hot    # equivalent to embedding_table[2]
print(f'Looked-up embedding: {looked_up}')
print(f'Direct index:        {embedding_table[2]}')

Scalar (learning rate): 0.0003
Token embedding (d=4): [ 0.2 -0.5  0.8  0.1]
Embedding table shape: (5, 4)
Looked-up embedding: [-0.46947439  0.54256004 -0.46341769 -0.46572975]
Direct index:        [-0.46947439  0.54256004 -0.46341769 -0.46572975]


## 2. Dot Product & Cosine Similarity

The **dot product** is the core operation inside attention: it measures how
aligned two vectors are. **Cosine similarity** normalises for magnitude and
is used in semantic search.

In [ ]:
# ── Dot Product and Cosine Similarity ────────────────────────────────────────

# Two word embeddings (simplified to 3-D for illustration)
king   = np.array([0.9,  0.1, 0.8])   # embedding for 'king'
queen  = np.array([0.85, 0.2, 0.75])  # embedding for 'queen' — similar
banana = np.array([0.1, -0.9, 0.05])  # embedding for 'banana' — different

# Raw dot product: larger when vectors point in the same direction
dot_kq = np.dot(king, queen)   # should be high
dot_kb = np.dot(king, banana)  # should be low / negative
print(f'Dot(king, queen):  {dot_kq:.4f}')
print(f'Dot(king, banana): {dot_kb:.4f}')

def cosine_similarity(a, b):
    """Normalised dot product — always in [-1, 1]."""
    # Divide by the product of L2 norms to remove magnitude effect
    return np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b))

print(f'CosSim(king, queen):  {cosine_similarity(king, queen):.4f}')   # ~1
print(f'CosSim(king, banana): {cosine_similarity(king, banana):.4f}')  # ~0 or negative

# ── Visualise in 2-D (projection) ─────────────────────────────────────────
fig, ax = plt.subplots(figsize=(5, 4))
for name, vec, col in [('king', king, '#34d399'), ('queen', queen, '#60a5fa'), ('banana', banana, '#f472b6')]:
    # Only plot first two dimensions for visualisation
    ax.annotate('', xy=(vec[0], vec[1]), xytext=(0, 0),
                arrowprops=dict(arrowstyle='->', color=col, lw=2))
    ax.text(vec[0]+0.02, vec[1]+0.02, name, color=col, fontsize=11, fontweight='bold')
ax.set_xlim(-1.2, 1.2); ax.set_ylim(-1.2, 1.2)
ax.set_title('Word vectors in 2-D projection')
ax.axhline(0, color='gray', lw=0.5); ax.axvline(0, color='gray', lw=0.5)
plt.tight_layout(); plt.show()

## 3. Softmax & Log-Softmax

The **softmax** function converts raw logits (real numbers) into a
probability distribution over the vocabulary. **Log-softmax** is used
during training for numerical stability.

In [ ]:
# ── Softmax and Log-Softmax ───────────────────────────────────────────────────

def softmax(x):
    """Numerically stable softmax using the max-shift trick."""
    # Subtracting max(x) prevents overflow — does NOT change the output
    x_shifted = x - np.max(x)
    exp_x = np.exp(x_shifted)          # exponentiate
    return exp_x / exp_x.sum()         # normalise to sum to 1

def log_softmax(x):
    """Log-softmax — used directly in NLL / cross-entropy loss."""
    # log(softmax(x)) but computed in a single numerically stable pass
    x_shifted = x - np.max(x)
    return x_shifted - np.log(np.exp(x_shifted).sum())

# Simulated logits from the final linear layer (vocab size = 6)
logits = np.array([2.0, 1.0, 0.1, -1.0, 0.5, 3.2])

probs     = softmax(logits)
log_probs = log_softmax(logits)

print('Logits:      ', logits)
print('Probabilities:', np.round(probs, 4))
print('Sum of probs: ', probs.sum())          # must be exactly 1.0
print('Log probs:   ', np.round(log_probs, 4))

# ── Visualise ─────────────────────────────────────────────────────────────────
tokens = [f'tok{i}' for i in range(len(logits))]
fig, axes = plt.subplots(1, 2, figsize=(10, 3))
axes[0].bar(tokens, logits, color='#60a5fa')
axes[0].set_title('Raw Logits'); axes[0].set_ylabel('Value')
axes[1].bar(tokens, probs, color='#34d399')
axes[1].set_title('After Softmax (probability)'); axes[1].set_ylabel('Probability')
plt.tight_layout(); plt.show()

## 4. Cross-Entropy Loss

**Cross-entropy** is the training objective for language modelling.
For each position, we maximise the log-probability of the correct next token.

In [ ]:
# ── Cross-Entropy Loss ────────────────────────────────────────────────────────

def cross_entropy_loss(logits, target_idx):
    """
    Cross-entropy loss for a single position.
    logits     : raw scores for each vocab token (1-D array)
    target_idx : index of the correct next token
    Returns    : scalar loss (lower is better)
    """
    # log-softmax gives us log P(token | context) for every token
    log_probs = log_softmax(logits)
    # NLL loss: negate the log-probability of the correct token
    return -log_probs[target_idx]

# Example: model predicts logits, correct token is index 5
target = 5
loss = cross_entropy_loss(logits, target)
print(f'Cross-entropy loss (target={target}): {loss:.4f}')
print(f'Equivalent: -log P(correct) = -log({probs[target]:.4f}) = {-np.log(probs[target]):.4f}')

# ── How loss changes with confidence ──────────────────────────────────────────
# As the model assigns higher probability to the correct token, loss decreases
p_correct = np.linspace(0.01, 0.99, 200)   # range of probabilities for correct token
nll = -np.log(p_correct)                   # negative log-likelihood

plt.figure(figsize=(6, 3))
plt.plot(p_correct, nll, color='#f59e0b', lw=2)
plt.xlabel('P(correct token)'); plt.ylabel('Cross-Entropy Loss')
plt.title('Lower probability → higher loss')
plt.grid(True, alpha=0.3); plt.tight_layout(); plt.show()

## 5. Scaled Dot-Product Attention

This is the mathematical core of the Transformer.
Given queries Q, keys K, and values V:

```
Attention(Q, K, V) = softmax( Q Kᵀ / √d_k ) V
```

The √d_k scaling prevents vanishing gradients in high dimensions.

In [ ]:
# ── Scaled Dot-Product Attention ─────────────────────────────────────────────

def scaled_dot_product_attention(Q, K, V, mask=None):
    """
    Full implementation of scaled dot-product attention.
    Q : (seq_len, d_k) — query matrix
    K : (seq_len, d_k) — key matrix
    V : (seq_len, d_v) — value matrix
    mask : optional causal mask — sets future positions to -inf
    Returns attention output (seq_len, d_v) and attention weights
    """
    d_k = Q.shape[-1]                        # key/query dimension

    # Step 1: compute raw attention scores — shape (seq_len, seq_len)
    scores = Q @ K.T                         # dot product of every query with every key

    # Step 2: scale by sqrt(d_k) to keep gradients well-behaved
    scores = scores / np.sqrt(d_k)

    # Step 3: apply causal mask (decoder uses this to prevent looking ahead)
    if mask is not None:
        scores = np.where(mask == 0, -1e9, scores)  # -1e9 ≈ -inf → zero after softmax

    # Step 4: softmax over the key dimension → attention weights
    attn_weights = np.array([softmax(row) for row in scores])  # (seq, seq)

    # Step 5: weighted sum of values
    output = attn_weights @ V                 # (seq_len, d_v)
    return output, attn_weights

# ── Demo: 4 tokens, d_k = d_v = 8 ────────────────────────────────────────────
seq_len, d_k, d_v = 4, 8, 8
Q = np.random.randn(seq_len, d_k)   # queries
K = np.random.randn(seq_len, d_k)   # keys
V = np.random.randn(seq_len, d_v)   # values

# Causal mask: upper triangle is 0 (masked), lower triangle + diagonal is 1
causal_mask = np.tril(np.ones((seq_len, seq_len)))

output, weights = scaled_dot_product_attention(Q, K, V, mask=causal_mask)
print(f'Attention output shape: {output.shape}')   # (4, 8)
print(f'Attention weights:\n{np.round(weights, 3)}')

# ── Visualise attention weights ───────────────────────────────────────────────
plt.figure(figsize=(4, 3.5))
plt.imshow(weights, cmap='Greens', vmin=0, vmax=1)
plt.colorbar(label='Attention weight')
plt.xlabel('Key position'); plt.ylabel('Query position')
plt.title('Causal attention weights')
plt.xticks(range(seq_len)); plt.yticks(range(seq_len))
plt.tight_layout(); plt.show()

## 6. Layer Normalisation

**LayerNorm** stabilises training by normalising each token's embedding
to zero mean and unit variance, then rescaling with learnable γ and β.

In [ ]:
# ── Layer Normalisation ───────────────────────────────────────────────────────

def layer_norm(x, gamma=None, beta=None, eps=1e-5):
    """
    Layer normalisation applied to the last dimension.
    x     : (d,) or (seq, d) — activations
    gamma : learnable scale parameter, shape (d,) — defaults to ones
    beta  : learnable shift parameter, shape (d,) — defaults to zeros
    eps   : small constant to prevent division by zero
    """
    d = x.shape[-1]
    if gamma is None: gamma = np.ones(d)   # initialised to 1
    if beta  is None: beta  = np.zeros(d)  # initialised to 0

    mean = x.mean(axis=-1, keepdims=True)  # per-token mean
    var  = x.var( axis=-1, keepdims=True)  # per-token variance

    # Normalise to N(0,1), then rescale and shift
    x_norm = (x - mean) / np.sqrt(var + eps)
    return gamma * x_norm + beta

# Simulate activations for a batch of 3 tokens, d_model=6
activations = np.array([[1.0, 2.0, 10.0, -3.0, 0.5, 7.0],   # large spread
                         [0.1, 0.2,  0.15,  0.3, 0.1, 0.2],  # small values
                         [5.0, 5.1,  4.9,   5.0, 5.2, 4.8]]) # tight cluster

normed = layer_norm(activations)
print('Before LayerNorm — mean/std per token:')
print('  mean:', activations.mean(axis=1).round(3))
print('  std: ', activations.std(axis=1).round(3))

print('After LayerNorm — mean/std per token:')
print('  mean:', normed.mean(axis=1).round(3))  # ~0
print('  std: ', normed.std(axis=1).round(3))   # ~1

## 7. Positional Encoding (Sinusoidal)

Transformers have no built-in sense of position. Sinusoidal encodings
add position information by injecting sin/cos waves of different frequencies.

In [ ]:
# ── Sinusoidal Positional Encoding ────────────────────────────────────────────

def positional_encoding(max_seq_len, d_model):
    """
    Compute sinusoidal positional encodings (Vaswani et al. 2017).
    max_seq_len : maximum sequence length
    d_model     : embedding dimension (must be even)
    Returns PE matrix of shape (max_seq_len, d_model)
    """
    # Create position indices [0, 1, ..., max_seq_len-1]
    positions = np.arange(max_seq_len)[:, None]   # shape (seq, 1)

    # Division terms: 10000^(2i/d_model) for each dimension pair i
    i = np.arange(0, d_model, 2)[None, :]         # even indices: 0,2,4,...
    div_term = np.power(10000, i / d_model)        # shape (1, d_model/2)

    PE = np.zeros((max_seq_len, d_model))
    PE[:, 0::2] = np.sin(positions / div_term)    # even dims → sine
    PE[:, 1::2] = np.cos(positions / div_term)    # odd dims  → cosine
    return PE

# Generate PE for a sequence of length 50, d_model=64
PE = positional_encoding(50, 64)
print(f'PE shape: {PE.shape}')  # (50, 64)

# ── Visualise ─────────────────────────────────────────────────────────────────
plt.figure(figsize=(10, 4))
plt.imshow(PE.T, cmap='RdBu', aspect='auto', origin='lower')
plt.colorbar(label='Value')
plt.xlabel('Token position'); plt.ylabel('Embedding dimension')
plt.title('Sinusoidal Positional Encoding (d_model=64, seq=50)')
plt.tight_layout(); plt.show()

# The first 4 dimensions show the frequency pattern clearly
plt.figure(figsize=(10, 3))
for dim, col in zip([0, 1, 2, 3], ['#34d399', '#60a5fa', '#a78bfa', '#f59e0b']):
    plt.plot(PE[:, dim], label=f'dim {dim}', color=col)
plt.legend(); plt.xlabel('Position'); plt.ylabel('PE value')
plt.title('Positional encoding — first 4 dimensions')
plt.grid(True, alpha=0.3); plt.tight_layout(); plt.show()

## 8. Gradient Descent Intuition

**Gradient descent** iteratively adjusts model weights to minimise the
loss. In LLMs, we use **AdamW** — a sophisticated variant with momentum,
adaptive learning rates, and weight decay.

In [ ]:
# ── Gradient Descent on a Toy Loss Surface ────────────────────────────────────

# Loss surface: L(w) = (w - 3)^2  — minimum at w=3
def loss_fn(w):
    """Simple quadratic loss with minimum at w=3."""
    return (w - 3.0) ** 2

def gradient(w):
    """Analytical gradient: dL/dw = 2(w - 3)."""
    return 2.0 * (w - 3.0)

# Hyperparameters
lr = 0.1          # learning rate — step size per update
w  = -2.0         # initial weight — far from minimum

# Track the optimisation path
history_w    = [w]
history_loss = [loss_fn(w)]

for step in range(30):
    grad = gradient(w)       # compute gradient at current w
    w = w - lr * grad        # move opposite to the gradient
    history_w.append(w)
    history_loss.append(loss_fn(w))

print(f'Final w: {w:.6f} (should be ~3.0)')
print(f'Final loss: {loss_fn(w):.2e}')

# ── Plot ──────────────────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(10, 3))
w_range = np.linspace(-3, 7, 200)
axes[0].plot(w_range, loss_fn(w_range), color='#94a3b8', label='Loss surface')
axes[0].plot(history_w, loss_fn(np.array(history_w)), 'o-', color='#34d399', ms=4, label='GD path')
axes[0].set_xlabel('w'); axes[0].set_ylabel('Loss'); axes[0].set_title('Gradient Descent path')
axes[0].legend(); axes[0].grid(True, alpha=0.3)
axes[1].semilogy(history_loss, color='#f59e0b')
axes[1].set_xlabel('Step'); axes[1].set_ylabel('Loss (log scale)')
axes[1].set_title('Loss vs step'); axes[1].grid(True, alpha=0.3)
plt.tight_layout(); plt.show()

## 9. Temperature Sampling

**Temperature** controls how creative or conservative text generation is.
- T < 1 : sharper distribution → more deterministic
- T = 1 : standard softmax
- T > 1 : flatter distribution → more random / creative

In [ ]:
# ── Temperature Sampling ──────────────────────────────────────────────────────

def sample_with_temperature(logits, temperature=1.0, n_samples=5000):
    """
    Apply temperature scaling then sample from the resulting distribution.
    logits      : raw model output scores
    temperature : T > 0; lower = more peaked; higher = more uniform
    n_samples   : number of samples to draw for demonstration
    """
    # Divide logits by temperature before softmax
    scaled_logits = logits / temperature
    probs = softmax(scaled_logits)
    # Multinomial sampling: draw token indices according to the distribution
    samples = np.random.choice(len(probs), size=n_samples, p=probs)
    return probs, samples

# Base logits for 6 vocabulary tokens
base_logits = np.array([3.0, 1.5, 0.5, -0.5, -1.0, 2.5])
tokens = [f'tok{i}' for i in range(6)]
temperatures = [0.2, 1.0, 2.0]

fig, axes = plt.subplots(1, 3, figsize=(13, 3), sharey=True)
for ax, T in zip(axes, temperatures):
    probs, _ = sample_with_temperature(base_logits, temperature=T)
    ax.bar(tokens, probs, color='#60a5fa', alpha=0.85)
    ax.set_title(f'Temperature = {T}')
    ax.set_xlabel('Token')
    ax.set_ylabel('Probability')
    ax.set_ylim(0, 1)
plt.suptitle('Effect of temperature on token probability distribution', y=1.02)
plt.tight_layout(); plt.show()

## 10. Perplexity

**Perplexity** (PPL) is the standard evaluation metric for language models.
Lower perplexity = better model.

```
PPL = exp( -1/N × Σ log P(token_i) )
```

In [ ]:
# ── Perplexity ────────────────────────────────────────────────────────────────

def perplexity(log_probs):
    """
    Compute perplexity from a sequence of log-probabilities.
    log_probs : array of log P(token_i | context) for each token in the sequence
    Returns   : scalar perplexity (lower is better)
    """
    # Average negative log-likelihood, then exponentiate
    avg_nll = -np.mean(log_probs)
    return np.exp(avg_nll)

# Scenario 1: confident model assigns high probability to every token
confident_log_probs  = np.array([-0.1, -0.2, -0.15, -0.1, -0.3])  # log P ≈ -0.17 avg

# Scenario 2: uncertain model — lower probabilities
uncertain_log_probs  = np.array([-2.5, -3.0, -2.8, -3.5, -2.0])

# Scenario 3: perfect model — assigns prob=1 to every correct token
perfect_log_probs    = np.zeros(5)   # log(1) = 0

for name, lp in [('Perfect model', perfect_log_probs),
                  ('Confident model', confident_log_probs),
                  ('Uncertain model', uncertain_log_probs)]:
    ppl = perplexity(lp)
    print(f'{name:20s} → PPL = {ppl:.2f}')

# GPT-2 small has PPL ~29 on Penn Treebank; GPT-4 level models ~5-8
print('\n(GPT-2 small ≈ PPL 29 on Penn Treebank; GPT-4 class ≈ PPL 5-8)')

## Summary

| Concept | Formula | Role in LLM |
|---|---|---|
| Dot product | a·b = Σ aᵢbᵢ | Attention score computation |
| Softmax | eˣⁱ / Σ eˣʲ | Convert logits to probabilities |
| Cross-entropy | -log P(correct) | Training loss |
| Scaled attention | softmax(QKᵀ/√d)V | Core attention mechanism |
| LayerNorm | (x-μ)/σ · γ + β | Training stability |
| Positional encoding | sin/cos at different freqs | Inject position information |
| Temperature | logits / T | Control generation diversity |
| Perplexity | exp(-1/N Σ log P) | Evaluation metric |